In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder \
    .appName("ecommerce") \
    .getOrCreate()

25/04/16 16:43:48 WARN Utils: Your hostname, brempong-HP-EliteBook-840-G7-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.36.43 instead (on interface wlp0s20f3)
25/04/16 16:43:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/16 16:43:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

In [4]:
df = spark.read.option("header", "true").csv("../scripts/Data/order_items_apr_2025.csv").withColumn("order_time", date_format(col("order_timestamp"), "HH:mm:ss"))
df.show()

+---+--------+-------+----------------------+----------+-----------------+---------+-------------------+----------+----------+
| id|order_id|user_id|days_since_prior_order|product_id|add_to_cart_order|reordered|    order_timestamp|      date|order_time|
+---+--------+-------+----------------------+----------+-----------------+---------+-------------------+----------+----------+
|  1|   10000|   1990|                    10|       988|                1|        0|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  2|   10000|   1990|                    10|       659|                2|        1|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  3|   10000|   1990|                    22|       676|                3|        0|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  4|   10000|   1990|                    14|         1|                4|        0|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  5|   10000|   1990|                    23|       907|                5|        0|2025-04-01T11:27:00|2025-04

In [5]:
#check for missing values
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+---+--------+-------+----------------------+----------+-----------------+---------+---------------+----+----------+
| id|order_id|user_id|days_since_prior_order|product_id|add_to_cart_order|reordered|order_timestamp|date|order_time|
+---+--------+-------+----------------------+----------+-----------------+---------+---------------+----+----------+
|  0|       0|      0|                     0|         0|                0|        0|              0|   0|         0|
+---+--------+-------+----------------------+----------+-----------------+---------+---------------+----+----------+



In [6]:
df = df.withColumn("reordered", when(col("reordered") == 1, "Reorder").otherwise("Not_Reorder"))
df.show()


+---+--------+-------+----------------------+----------+-----------------+-----------+-------------------+----------+----------+
| id|order_id|user_id|days_since_prior_order|product_id|add_to_cart_order|  reordered|    order_timestamp|      date|order_time|
+---+--------+-------+----------------------+----------+-----------------+-----------+-------------------+----------+----------+
|  1|   10000|   1990|                    10|       988|                1|Not_Reorder|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  2|   10000|   1990|                    10|       659|                2|    Reorder|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  3|   10000|   1990|                    22|       676|                3|Not_Reorder|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  4|   10000|   1990|                    14|         1|                4|Not_Reorder|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  5|   10000|   1990|                    23|       907|                5|Not_Reorder|2025-04-01T

In [7]:
df.select("date").distinct().show()

+----------+
|      date|
+----------+
|2025-04-01|
+----------+



In [8]:
sorted = df.repartition(2, col("reordered")).sortWithinPartitions(col("date"))
sorted.show()

+---+--------+-------+----------------------+----------+-----------------+-----------+-------------------+----------+----------+
| id|order_id|user_id|days_since_prior_order|product_id|add_to_cart_order|  reordered|    order_timestamp|      date|order_time|
+---+--------+-------+----------------------+----------+-----------------+-----------+-------------------+----------+----------+
|  1|   10000|   1990|                    10|       988|                1|Not_Reorder|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  2|   10000|   1990|                    10|       659|                2|    Reorder|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  3|   10000|   1990|                    22|       676|                3|Not_Reorder|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  4|   10000|   1990|                    14|         1|                4|Not_Reorder|2025-04-01T11:27:00|2025-04-01|  11:27:00|
|  5|   10000|   1990|                    23|       907|                5|Not_Reorder|2025-04-01T

In [ ]:
products = spark.read.option("header", "true").csv("../scripts/Data/products.csv")
products.show()


+----------+-------------+-----------+-------------------+
|product_id|department_id| department|       product_name|
+----------+-------------+-----------+-------------------+
|         1|            4|      Books|    Product_1_Store|
|         2|            2|      Books|    Product_2_There|
|         3|            4|      Books|     Product_3_Hand|
|         4|            6|     Sports|       Product_4_Tv|
|         5|            1|       Toys|     Product_5_Easy|
|         6|            1|     Sports|    Product_6_Woman|
|         7|            5|       Home|     Product_7_Yard|
|         8|            5|      Books|  Product_8_Manager|
|         9|            6|       Toys|     Product_9_Face|
|        10|            4|     Sports|   Product_10_Sport|
|        11|            2|   Clothing|  Product_11_Parent|
|        12|            2|      Books| Product_12_Contain|
|        13|            4|     Sports|Product_13_Audience|
|        14|            1|       Home|     Product_14_Jo

In [13]:
products.select("department").distinct().show()

+-----------+
| department|
+-----------+
|       Home|
|     Sports|
|Electronics|
|   Clothing|
|      Books|
|       Toys|
+-----------+



In [15]:
partioned_product = products.repartition(6, col("department"))
partioned_product.show()

+----------+-------------+----------+--------------------+
|product_id|department_id|department|        product_name|
+----------+-------------+----------+--------------------+
|         1|            4|     Books|     Product_1_Store|
|         2|            2|     Books|     Product_2_There|
|         3|            4|     Books|      Product_3_Hand|
|         7|            5|      Home|      Product_7_Yard|
|         8|            5|     Books|   Product_8_Manager|
|        12|            2|     Books|  Product_12_Contain|
|        14|            1|      Home|      Product_14_Job|
|        26|            1|      Home|Product_26_Develo...|
|        29|            4|      Home|    Product_29_Cause|
|        30|            1|      Home| Product_30_Thousand|
|        31|            3|     Books|       Product_31_My|
|        32|            6|      Home| Product_32_Together|
|        33|            6|     Books|     Product_33_Card|
|        43|            6|     Books|Product_43_Confer..

In [11]:
orders = spark.read.option("header", "true").csv("../scripts/Data/orders_apr_2025.csv").withColumn("order_time", date_format(col("order_timestamp"), "HH:mm:ss"))
orders.show()

+---------+--------+-------+-------------------+------------+----------+----------+
|order_num|order_id|user_id|    order_timestamp|total_amount|      date|order_time|
+---------+--------+-------+-------------------+------------+----------+----------+
|       90|   10000|   1990|2025-04-01T11:27:00|      229.53|2025-04-01|  11:27:00|
|       41|   10001|   5057|2025-04-01T17:53:00|      131.93|2025-04-01|  17:53:00|
|       22|   10002|   7864|2025-04-01T02:26:00|       251.9|2025-04-01|  02:26:00|
|       99|   10003|   3131|2025-04-01T01:24:00|      487.49|2025-04-01|  01:24:00|
|       51|   10004|   9621|2025-04-01T11:48:00|      365.46|2025-04-01|  11:48:00|
|       86|   10005|   8777|2025-04-01T18:43:00|       84.22|2025-04-01|  18:43:00|
|       42|   10006|   2135|2025-04-01T21:33:00|       380.7|2025-04-01|  21:33:00|
|       30|   10007|   7672|2025-04-01T04:10:00|      464.95|2025-04-01|  04:10:00|
|       43|   10008|   1941|2025-04-01T19:06:00|      332.11|2025-04-01|  19

In [16]:
orders_part = orders.repartition(1, col("date"))
orders_part.show()

+---------+--------+-------+-------------------+------------+----------+----------+
|order_num|order_id|user_id|    order_timestamp|total_amount|      date|order_time|
+---------+--------+-------+-------------------+------------+----------+----------+
|       90|   10000|   1990|2025-04-01T11:27:00|      229.53|2025-04-01|  11:27:00|
|       41|   10001|   5057|2025-04-01T17:53:00|      131.93|2025-04-01|  17:53:00|
|       22|   10002|   7864|2025-04-01T02:26:00|       251.9|2025-04-01|  02:26:00|
|       99|   10003|   3131|2025-04-01T01:24:00|      487.49|2025-04-01|  01:24:00|
|       51|   10004|   9621|2025-04-01T11:48:00|      365.46|2025-04-01|  11:48:00|
|       86|   10005|   8777|2025-04-01T18:43:00|       84.22|2025-04-01|  18:43:00|
|       42|   10006|   2135|2025-04-01T21:33:00|       380.7|2025-04-01|  21:33:00|
|       30|   10007|   7672|2025-04-01T04:10:00|      464.95|2025-04-01|  04:10:00|
|       43|   10008|   1941|2025-04-01T19:06:00|      332.11|2025-04-01|  19